# ClusterFewshot Demo: Bring-Your-Own-Encoder Examples

This notebook demonstrates how to use **ClusterFewshot** with different semantic encoders for various task types.

**ClusterFewshot** is a task-adaptive few-shot selection optimizer that:
1. Clusters examples semantically using custom encoders
2. Ranks examples by their one-shot demonstration impact
3. Selects optimal few-shot subsets using task-specific strategies

## Setup & Imports

In [ ]:
import dspy
from dspy.teleprompt.clusterfewshot import (
    ClusterFewshot,
    SemanticEncoder,
    create_sentence_transformer_encoder,
    create_numeric_encoder,
    sentence_transformer_transform,
)

# Enable experimental features
dspy.settings.experimental = True

print("✓ ClusterFewshot imported successfully!")

## Example 1: Simple Arithmetic Task (Dummy Data)

Let's start with a minimal example using dummy data to understand the basic workflow.

In [ ]:
# Create dummy arithmetic examples
dummy_train = [
    dspy.Example(question="What is 2 + 2?", answer="4").with_inputs("question"),
    dspy.Example(question="What is 5 - 3?", answer="2").with_inputs("question"),
    dspy.Example(question="What is 3 * 4?", answer="12").with_inputs("question"),
    dspy.Example(question="What is 10 / 2?", answer="5").with_inputs("question"),
    dspy.Example(question="What is 7 + 8?", answer="15").with_inputs("question"),
    dspy.Example(question="What is 20 - 5?", answer="15").with_inputs("question"),
    dspy.Example(question="What is 6 * 3?", answer="18").with_inputs("question"),
    dspy.Example(question="What is 100 / 10?", answer="10").with_inputs("question"),
]

dummy_val = [
    dspy.Example(question="What is 9 + 1?", answer="10").with_inputs("question"),
    dspy.Example(question="What is 8 - 2?", answer="6").with_inputs("question"),
    dspy.Example(question="What is 5 * 2?", answer="10").with_inputs("question"),
]

print(f"Training examples: {len(dummy_train)}")
print(f"Validation examples: {len(dummy_val)}")

In [ ]:
# Configure a dummy LM (using a mock for demonstration)
# In production, use: dspy.LM("openai/gpt-4o-mini") or your preferred model

class DummyLM:
    """Mock LM for demonstration purposes"""
    def __call__(self, prompt, **kwargs):
        # Simple pattern matching for demo
        if "2 + 2" in prompt:
            return ["4"]
        return ["42"]  # Default answer

lm = DummyLM()
dspy.configure(lm=lm)

print("✓ Dummy LM configured")

In [ ]:
# Define a simple Chain-of-Thought program
class SimpleCoT(dspy.Module):
    def __init__(self):
        super().__init__()
        self.predict = dspy.ChainOfThought("question -> answer")
    
    def forward(self, question):
        return self.predict(question=question)

student = SimpleCoT()
print("✓ Student program defined")

In [ ]:
# Create semantic encoders for text-based tasks
encoders = [
    create_sentence_transformer_encoder("sentence-transformers/all-mpnet-base-v2"),
    # Add more encoders for comparison:
    # create_sentence_transformer_encoder("sentence-transformers/gtr-t5-base"),
]

print(f"✓ Created {len(encoders)} semantic encoder(s)")
for enc in encoders:
    print(f"  - {enc}")

In [ ]:
# Define a simple metric
def simple_metric(example, prediction, trace=None):
    """Check if answer matches (case-insensitive)"""
    return str(example.answer).strip().lower() == str(prediction.answer).strip().lower()

print("✓ Metric defined")

In [ ]:
# Initialize ClusterFewshot optimizer
optimizer = ClusterFewshot(
    task_type="arithmetic",
    metric=simple_metric,
    semantic_encoders=encoders,
    apply_visuals=True,  # Enable visualizations
)

print("✓ ClusterFewshot optimizer initialized")
print(f"  Task type: {optimizer.task_type}")
print(f"  Encoders: {len(optimizer.semantic_encoders)}")
print(f"  Visualizations: {optimizer.apply_visuals}")

In [ ]:
# Compile the optimizer
# NOTE: This will take a few minutes depending on dataset size

optimized_program = optimizer.compile(
    student=student,
    trainset=dummy_train,
    valset=dummy_val
)

print("✓ Optimization complete!")
print(f"  Selected encoder: {optimizer.selected_encoder}")
print(f"  Number of clusters (K): {optimizer.N}")

In [ ]:
# Inspect the final few-shot subset
print(f"Final few-shot demonstrations ({len(optimizer.final_fewshot_subset)} total):\n")

for i, demo in enumerate(optimizer.final_fewshot_subset, 1):
    raw_ex = demo['raw']
    print(f"{i}. Q: {raw_ex.question}")
    print(f"   A: {raw_ex.answer}\n")

## Example 2: Custom SemanticEncoder

Create a custom encoder with a specialized transform function.

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

# Define a custom transform that extracts specific features
def custom_arithmetic_transform(encoder, examples):
    """
    Custom transform that combines question text with operation type detection
    """
    texts = []
    for ex in examples:
        q = ex.question
        # Detect operation type
        if '+' in q or 'plus' in q.lower() or 'add' in q.lower():
            op_type = "addition"
        elif '-' in q or 'minus' in q.lower() or 'subtract' in q.lower():
            op_type = "subtraction"
        elif '*' in q or 'times' in q.lower() or 'multiply' in q.lower():
            op_type = "multiplication"
        elif '/' in q or 'divide' in q.lower():
            op_type = "division"
        else:
            op_type = "unknown"
        
        # Combine question with operation type
        texts.append(f"{q} [Operation: {op_type}]")
    
    return encoder.encode(texts, convert_to_numpy=True)

# Create custom encoder
model = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")
custom_encoder = SemanticEncoder(
    encoder=model,
    transform_fn=custom_arithmetic_transform,
    name="CustomArithmeticEncoder"
)

print(f"✓ Custom encoder created: {custom_encoder}")

In [ ]:
# Use the custom encoder
optimizer_custom = ClusterFewshot(
    task_type="arithmetic",
    metric=simple_metric,
    semantic_encoders=[custom_encoder],
    apply_visuals=False,  # Disable for faster execution
)

optimized_custom = optimizer_custom.compile(
    student=SimpleCoT(),
    trainset=dummy_train,
    valset=dummy_val
)

print("✓ Custom encoder optimization complete!")

## Example 3: Classification with Numeric Encoder (Iris-style)

For classification tasks with numeric inputs, use the `create_numeric_encoder()`.

In [ ]:
# Create dummy classification data (Iris-style)
dummy_iris_train = [
    dspy.Example(sepal_length=5.1, sepal_width=3.5, petal_length=1.4, petal_width=0.2, species="setosa").with_inputs("sepal_length", "sepal_width", "petal_length", "petal_width"),
    dspy.Example(sepal_length=4.9, sepal_width=3.0, petal_length=1.4, petal_width=0.2, species="setosa").with_inputs("sepal_length", "sepal_width", "petal_length", "petal_width"),
    dspy.Example(sepal_length=7.0, sepal_width=3.2, petal_length=4.7, petal_width=1.4, species="versicolor").with_inputs("sepal_length", "sepal_width", "petal_length", "petal_width"),
    dspy.Example(sepal_length=6.4, sepal_width=3.2, petal_length=4.5, petal_width=1.5, species="versicolor").with_inputs("sepal_length", "sepal_width", "petal_length", "petal_width"),
    dspy.Example(sepal_length=6.3, sepal_width=3.3, petal_length=6.0, petal_width=2.5, species="virginica").with_inputs("sepal_length", "sepal_width", "petal_length", "petal_width"),
    dspy.Example(sepal_length=5.8, sepal_width=2.7, petal_length=5.1, petal_width=1.9, species="virginica").with_inputs("sepal_length", "sepal_width", "petal_length", "petal_width"),
]

dummy_iris_val = [
    dspy.Example(sepal_length=5.0, sepal_width=3.6, petal_length=1.4, petal_width=0.2, species="setosa").with_inputs("sepal_length", "sepal_width", "petal_length", "petal_width"),
    dspy.Example(sepal_length=6.5, sepal_width=2.8, petal_length=4.6, petal_width=1.5, species="versicolor").with_inputs("sepal_length", "sepal_width", "petal_length", "petal_width"),
]

print(f"Classification training examples: {len(dummy_iris_train)}")
print(f"Classification validation examples: {len(dummy_iris_val)}")

In [ ]:
# Define classification program
class IrisClassifier(dspy.Module):
    def __init__(self):
        super().__init__()
        self.predict = dspy.ChainOfThought(
            "sepal_length, sepal_width, petal_length, petal_width -> species"
        )
    
    def forward(self, sepal_length, sepal_width, petal_length, petal_width):
        return self.predict(
            sepal_length=sepal_length,
            sepal_width=sepal_width,
            petal_length=petal_length,
            petal_width=petal_width
        )

print("✓ Classification program defined")

In [ ]:
# Use numeric encoder for classification
numeric_encoder = create_numeric_encoder()

optimizer_iris = ClusterFewshot(
    task_type="classification",
    metric=lambda ex, pred, trace: ex.species == pred.species,
    semantic_encoders=[numeric_encoder],
    apply_visuals=False,
)

optimized_iris = optimizer_iris.compile(
    student=IrisClassifier(),
    trainset=dummy_iris_train,
    valset=dummy_iris_val
)

print("✓ Classification optimization complete!")
print(f"  Selected encoder: {optimizer_iris.selected_encoder}")
print(f"  Number of clusters: {optimizer_iris.N}")

## Example 4: Soft Selection (Advanced)

Use differentiable soft selection to balance quality and diversity.

In [ ]:
# Enable soft selection
optimizer_soft = ClusterFewshot(
    task_type="arithmetic",
    metric=simple_metric,
    semantic_encoders=encoders,
    soft_select=True,  # Enable differentiable selection
    apply_visuals=True,  # Will show diversity visualization
)

optimized_soft = optimizer_soft.compile(
    student=SimpleCoT(),
    trainset=dummy_train,
    valset=dummy_val
)

print("✓ Soft selection optimization complete!")
print("  Check 'soft_selection_pca.png' for diversity visualization")

## Example 5: Comparing Multiple Encoders

Let ClusterFewshot automatically select the best encoder via grid search.

In [ ]:
# Create multiple encoders for comparison
multi_encoders = [
    create_sentence_transformer_encoder("sentence-transformers/all-mpnet-base-v2"),
    create_sentence_transformer_encoder("sentence-transformers/gtr-t5-base"),
    # Add more encoders:
    # create_sentence_transformer_encoder("BAAI/bge-large-en-v1.5"),
]

print(f"Comparing {len(multi_encoders)} encoders:")
for enc in multi_encoders:
    print(f"  - {enc}")

In [ ]:
# Run optimization with encoder comparison
optimizer_multi = ClusterFewshot(
    task_type="arithmetic",
    metric=simple_metric,
    semantic_encoders=multi_encoders,
    apply_visuals=False,
)

optimized_multi = optimizer_multi.compile(
    student=SimpleCoT(),
    trainset=dummy_train,
    valset=dummy_val
)

print("\n✓ Multi-encoder optimization complete!")
print(f"  Best encoder selected: {optimizer_multi.selected_encoder}")
print(f"  Number of clusters: {optimizer_multi.N}")

## Summary & Next Steps

You've learned how to:
1. ✅ Use ClusterFewshot with default encoders
2. ✅ Create custom SemanticEncoders
3. ✅ Use numeric encoders for classification
4. ✅ Enable soft selection for diversity
5. ✅ Compare multiple encoders automatically

### Next Steps:

- **Try with real datasets**: GSM8K, HotPotQA, your custom data
- **Experiment with encoders**: Test different SentenceTransformer models
- **Tune parameters**: Adjust metric thresholds, visualization settings
- **Read the docs**: See `clusterfewshot.md` for complete API reference

### Resources:

- [DSPy Documentation](https://dspy-docs.vercel.app/)
- [ClusterFewshot API Reference](./clusterfewshot.md)
- [SentenceTransformers Models](https://www.sbert.net/docs/pretrained_models.html)